# gpt2 finetuned on wikitext

In [1]:
!pip install --quiet transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-c

In [2]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)


Mounted at /content/drive/


In [3]:
%cd drive/MyDrive/ColabNotebooks/NLP_HW_finetuning
!ls

/content/drive/MyDrive/ColabNotebooks/NLP_HW_finetuning
 distilgpt2+wiki.ipynb	 gpt2+wiki.ipynb		        wandb
 distillgpt2+ptb.ipynb	 models
 gpt2+ptb.ipynb		'NLP_Tutorial2_and_HW_Mali (1).ipynb'


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset

In [ ]:
# 1. Model and tokenizer initialization
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# for models like gpt-2 that don't have a pad token by default
tokenizer.pad_token = tokenizer.eos_token

# for other models without pad token
# tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# 2. Load model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt2_model_wiki = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# 3. Load Dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")


train_data = dataset["train"]
val_data = dataset['validation']

# 4. Tokenization Function
def tokenize_function(example):
  result = tokenizer(
                  example["text"],
                  truncation=True,
                  padding="max_length", # could also use ’longest’
                  max_length=512, # set a reasonable max length
                  )
  result["labels"] = result["input_ids"].copy()
  return result


# 5. Apply Tokenization and REmove Original 'text' Column
tokenized_train = train_data.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_val = val_data.map(tokenize_function, batched=True, remove_columns=['text'])

# Remove columns other than input_ids/attention_mask
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
# tokenized_train = tokenized_train.with_format("torch", device=device)
# tokenized_val = tokenized_val.with_format("torch", device=device)
# 7. Training Arguments
training_args = TrainingArguments(
                            output_dir="./gpt2_wiki",
                            eval_strategy="epoch",
                            save_strategy="epoch",
                            num_train_epochs=1, # Increase for better results
                            per_device_train_batch_size=8, # Adjust based on GPU memory
                            per_device_eval_batch_size=8,
                            save_steps=500,
                            logging_steps=100,
                            load_best_model_at_end=True,
                            remove_unused_columns=False,
                            push_to_hub=False
                            )

# 8. Trainer Definition
trainer = Trainer(
              model=gpt2_model_wiki,
              args=training_args,
              train_dataset=tokenized_train,
              eval_dataset=tokenized_val,
              tokenizer=tokenizer # tokenizer is important
              )

# 9. Training
trainer.train()


prompt = "In␣the␣future,␣we␣wish␣to␣learn␣NLP␣and␣develop␣novel␣artificial␣intelligence␣agents"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
# Generate text
output_ids = gpt2_model_wiki.generate(
                          input_ids,
                          max_length=512,
                          num_beams=5,
                          no_repeat_ngram_size=2,
                          early_stopping=True
                          )
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Prompt:", prompt)
print("Generated␣text:\n", generated_text)
model.save_pretrained("./models/gpt2-wiki")
tokenizer.save_pretrained("./models/gpt2-wiki")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hiteshuv (hiteshuv-university-of-south-florida) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
